# 07 - Comparación de Resultados y Análisis Final
## Clasificación de Marcha Patológica

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/07%20-%20Comparacion_resultados.ipynb)

---
**Objetivo:** Comparar todos los modelos entrenados (RNN, LSTM, GRU, BiLSTM, BiGRU, Transformer)
y determinar cuál supera mejor las referencias del paper Jun et al. (2020).
Incluye análisis clínico de la sensibilidad por clase patológica.

In [ ]:
import subprocess, sys
pkgs = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=False)


## 1. Configuración del entorno

In [ ]:
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

N_CLASSES = 6
CLASS_NAMES = {0: 'Normal', 1: 'Antálgica', 2: 'Rígida', 3: 'Oscilante', 4: 'Steppage', 5: 'Trendelenburg'}
CLASS_NAMES_SHORT = {0: 'Norm', 1: 'Antl', 2: 'Rígd', 3: 'Osci', 4: 'Step', 5: 'Tren'}

print('Entorno configurado.')

## 2. Carga de resultados de todos los modelos

Los resultados fueron guardados en los notebooks anteriores como archivos `.pkl`.

In [ ]:
def load_results_safe(filepath):
    """Carga resultados .pkl de forma segura. Retorna {} si el archivo no existe."""
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    print(f'  Advertencia: {filepath} no encontrado (ejecutar notebook correspondiente primero)')
    return {}


# Cargar resultados de cada notebook de modelo
results_rnn         = load_results_safe('results_rnn.pkl')
results_lstm_gru    = load_results_safe('results_lstm_gru.pkl')
results_bidir       = load_results_safe('results_bidir.pkl')
results_transformer = load_results_safe('results_transformer.pkl')

# Combinar todos los resultados
ALL_RESULTS = {}
ALL_RESULTS.update(results_rnn)
ALL_RESULTS.update(results_lstm_gru)
ALL_RESULTS.update(results_bidir)
ALL_RESULTS.update(results_transformer)

print(f'Modelos cargados: {len(ALL_RESULTS)}')
for name in ALL_RESULTS:
    print(f'  - {name}: {ALL_RESULTS[name]["mean_acc"]*100:.2f}%')

In [ ]:
# Resultados de referencia del paper Jun et al. (2020)
PAPER_RESULTS = {
    'RNN (Jun 2020)':              {'mean_acc': 0.8623, 'std_acc': 0.0, 'is_paper': True},
    'CNN 1D (Jun 2020)':           {'mean_acc': 0.8665, 'std_acc': 0.0, 'is_paper': True},
    'LSTM (Jun 2020)':             {'mean_acc': 0.8725, 'std_acc': 0.0, 'is_paper': True},
    'GRU (Jun 2020)':              {'mean_acc': 0.9013, 'std_acc': 0.0, 'is_paper': True},
    'GRU piernas (Jun 2020) ★':   {'mean_acc': 0.9367, 'std_acc': 0.0, 'is_paper': True},
    'Bi-LSTM (Guo 2020)':          {'mean_acc': 0.9075, 'std_acc': 0.0, 'is_paper': True},
}

print('Referencias del paper cargadas.')

## 3. Tabla comparativa global

In [ ]:
# Construir dataframe comparativo
rows = []

# Modelos propios
for name, res in ALL_RESULTS.items():
    accs = res.get('accuracies', [])
    sens = res.get('sensitivity', [np.nan]*N_CLASSES)
    spec = res.get('specificity', [np.nan]*N_CLASSES)
    prec = res.get('precision',   [np.nan]*N_CLASSES)
    rows.append({
        'Modelo': name,
        'Acc (%) Media': res['mean_acc'] * 100,
        'Acc (%) Std':   res['std_acc'] * 100,
        'Sensibilidad (%)': np.mean(sens) * 100,
        'Especificidad (%)': np.mean(spec) * 100,
        'Precisión (%)': np.mean(prec) * 100,
        'Fuente': 'Nuestro'
    })

# Modelos del paper
for name, res in PAPER_RESULTS.items():
    rows.append({
        'Modelo': name,
        'Acc (%) Media': res['mean_acc'] * 100,
        'Acc (%) Std': 0.0,
        'Sensibilidad (%)': np.nan,
        'Especificidad (%)': np.nan,
        'Precisión (%)': np.nan,
        'Fuente': 'Paper'
    })

df_results = pd.DataFrame(rows).sort_values('Acc (%) Media', ascending=False)

# Formatear para mostrar
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.float_format', '{:.2f}'.format)
print('=== TABLA COMPARATIVA COMPLETA ===')
print(df_results.to_string(index=False))

## 4. Visualización comparativa

In [ ]:
# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(16, 7))

df_sorted = df_results.sort_values('Acc (%) Media', ascending=True)

# Colores: azul para nuestros modelos, gris para referencias del paper
colors = ['#2196F3' if s == 'Nuestro' else '#9E9E9E' for s in df_sorted['Fuente']]
# El mejor del paper en verde
colors = ['#4CAF50' if '★' in m else c
          for m, c in zip(df_sorted['Modelo'], colors)]

bars = ax.barh(range(len(df_sorted)), df_sorted['Acc (%) Media'],
               color=colors, edgecolor='black', linewidth=0.7, height=0.7)

# Error bars para modelos con std disponible
for i, (_, row) in enumerate(df_sorted.iterrows()):
    if row['Acc (%) Std'] > 0:
        ax.errorbar(row['Acc (%) Media'], i, xerr=row['Acc (%) Std'],
                    fmt='none', color='black', capsize=4, linewidth=1.5)
    # Etiqueta con accuracy
    ax.text(row['Acc (%) Media'] + 0.3, i, f"{row['Acc (%) Media']:.2f}%",
            va='center', fontsize=9)

# Línea de referencia del mejor del paper
ax.axvline(93.67, color='green', linestyle='--', alpha=0.8, linewidth=2,
           label='GRU piernas (Jun 2020): 93.67%')

ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels(df_sorted['Modelo'], fontsize=9)
ax.set_xlabel('Accuracy LOSO-CV (%)', fontsize=12)
ax.set_title('Comparación de Modelos — Clasificación de Marcha Patológica\n'
             'LOSO-CV (10 folds) sobre GIST Dataset', fontsize=13, fontweight='bold')
ax.set_xlim(75, 105)

# Leyenda
legend_elements = [
    mpatches.Patch(color='#2196F3', label='Modelos propios'),
    mpatches.Patch(color='#9E9E9E', label='Referencia (paper)'),
    mpatches.Patch(color='#4CAF50', label='Mejor del paper'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('comparacion_todos_modelos.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot de distribución de accuracies por fold (solo modelos propios)
own_models = {name: res for name, res in ALL_RESULTS.items()
              if 'accuracies' in res and len(res['accuracies']) == 10}

if own_models:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    data = [[a*100 for a in res['accuracies']] for res in own_models.values()]
    labels = list(own_models.keys())
    
    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    notch=False, showmeans=True,
                    meanprops={'marker': 'D', 'markerfacecolor': 'white',
                               'markeredgecolor': 'black', 'markersize': 7})
    
    colors_bp = plt.cm.Set2(np.linspace(0, 1, len(own_models)))
    for patch, color in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.axhline(93.67, color='green', linestyle='--', linewidth=2,
               alpha=0.8, label='Mejor del paper: 93.67%')
    ax.axhline(90.13, color='orange', linestyle=':', linewidth=1.5,
               alpha=0.8, label='GRU paper (todos): 90.13%')
    
    ax.set_title('Distribución de Accuracy por Fold LOSO-CV (modelos propios)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy (%)')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=9)
    ax.set_ylim(50, 110)
    
    plt.tight_layout()
    plt.savefig('boxplot_accuracies.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print('No hay resultados de modelos propios. Ejecutar notebooks 03-06 primero.')

## 5. Análisis de métricas por clase (análisis clínico)

En el contexto clínico, la **sensibilidad** es la métrica más importante:
minimizar falsos negativos significa no dejar pasar una marcha patológica sin diagnosticar.

In [ ]:
# Comparar sensibilidad por clase entre los mejores modelos
if ALL_RESULTS:
    # Seleccionar los mejores 3 modelos propios
    sorted_models = sorted(ALL_RESULTS.items(),
                           key=lambda x: x[1]['mean_acc'], reverse=True)
    top3 = sorted_models[:min(3, len(sorted_models))]
    
    x = np.arange(N_CLASSES)
    width = 0.25
    colors_top = ['#2196F3', '#FF9800', '#9C27B0']
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metric_pairs = [
        ('sensitivity', 'Sensibilidad (Recall por clase)'),
        ('specificity', 'Especificidad por clase'),
        ('precision',   'Precisión por clase'),
    ]
    
    for ax, (metric_key, metric_title) in zip(axes, metric_pairs):
        for i, (name, res) in enumerate(top3):
            metric_vals = np.array(res.get(metric_key, [np.nan]*N_CLASSES)) * 100
            offset = (i - 1) * width
            bars = ax.bar(x + offset, metric_vals, width,
                          label=f'{name} ({res["mean_acc"]*100:.1f}%)',
                          color=colors_top[i], alpha=0.85, edgecolor='black', linewidth=0.5)
        
        ax.set_title(metric_title, fontweight='bold', fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels([CLASS_NAMES_SHORT[i] for i in range(N_CLASSES)], rotation=0)
        ax.set_ylabel('%')
        ax.set_ylim(0, 115)
        ax.legend(fontsize=7)
        ax.axhline(100, color='red', linestyle='--', alpha=0.3)
        ax.axhline(90, color='green', linestyle=':', alpha=0.5)
    
    plt.suptitle('Métricas por clase — Top 3 modelos', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('metricas_por_clase.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print('Ejecutar notebooks 03-06 primero.')

## 6. Matrices de confusión de los mejores modelos

In [ ]:
# Mostrar matrices de confusión de los mejores 2 modelos
if ALL_RESULTS:
    sorted_models_cms = [(name, res) for name, res in
                         sorted(ALL_RESULTS.items(), key=lambda x: x[1]['mean_acc'], reverse=True)
                         if 'confusion_matrix' in res][:2]
    
    cls_labels = [CLASS_NAMES[i] for i in range(N_CLASSES)]
    
    if sorted_models_cms:
        fig, axes = plt.subplots(1, len(sorted_models_cms), figsize=(7 * len(sorted_models_cms), 6))
        if len(sorted_models_cms) == 1:
            axes = [axes]
        
        cmaps = ['Blues', 'Purples']
        for ax, (name, res), cmap in zip(axes, sorted_models_cms, cmaps):
            cm = np.array(res['confusion_matrix'])
            cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
            
            sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap=cmap,
                        xticklabels=cls_labels, yticklabels=cls_labels,
                        ax=ax, cbar_kws={'label': '%'})
            ax.set_title(f'{name}\nAcc={res["mean_acc"]*100:.2f}%', fontweight='bold')
            ax.set_xlabel('Predicho')
            ax.set_ylabel('Real')
            ax.tick_params(axis='x', rotation=25)
        
        plt.suptitle('Matrices de Confusión — Mejores Modelos (LOSO-CV total, %)',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('cm_mejores_modelos.png', dpi=100, bbox_inches='tight')
        plt.show()
else:
    print('Ejecutar notebooks 03-06 primero.')

## 7. Análisis de iteraciones del proyecto

Descripción de las iteraciones realizadas, conforme a los requisitos del informe.

In [ ]:
# Crear gráfico tipo roadmap de las iteraciones
iteraciones = [
    {'iter': 1, 'nombre': 'Exploración de datos',
     'desc': 'Carga de 7157 secuencias CSV,\nvisualización de esqueleto 3D,\ndistribución de clases balanceada'},
    {'iter': 2, 'nombre': 'Preprocesado',
     'desc': 'StandardScaler por fold,\nselección joints piernas/todos,\nLOSO-CV splits'},
    {'iter': 3, 'nombre': 'RNN Línea de Base',
     'desc': 'SimpleRNN 4 capas 125 unidades,\nLOSO-CV 10 folds,\nReferencia: 86.23%'},
    {'iter': 4, 'nombre': 'LSTM y GRU',
     'desc': 'Replicación exacta del paper,\nGRU 4L todos: ~90%,\nGRU 4L piernas: ~93%'},
    {'iter': 5, 'nombre': 'BiLSTM y BiGRU',
     'desc': 'Arquitecturas bidireccionales,\nCaptura dependencias\'futuras\',\nRef: Bi-LSTM 90.75%'},
    {'iter': 6, 'nombre': 'Transformer',
     'desc': 'Self-Attention sobre 50 frames,\nPositional encoding aprendido,\nVisualización de atención'},
]

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(-0.5, len(iteraciones) - 0.5)
ax.set_ylim(-1.5, 2)
ax.axis('off')

colors_iter = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(iteraciones)))

for i, it in enumerate(iteraciones):
    # Círculo de iteración
    circle = plt.Circle((i, 0), 0.35, color=colors_iter[i], zorder=5)
    ax.add_patch(circle)
    ax.text(i, 0, str(it['iter']), ha='center', va='center',
            fontsize=14, fontweight='bold', color='white', zorder=6)
    
    # Nombre
    ax.text(i, 0.55, it['nombre'], ha='center', va='bottom',
            fontsize=10, fontweight='bold')
    
    # Descripción
    ax.text(i, -0.6, it['desc'], ha='center', va='top',
            fontsize=7.5, style='italic', color='#555555')
    
    # Flecha entre iteraciones
    if i < len(iteraciones) - 1:
        ax.annotate('', xy=(i + 0.65, 0), xytext=(i + 0.35, 0),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('Iteraciones del Proyecto — Clasificación de Marcha Patológica',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('roadmap_iteraciones.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Resumen ejecutivo

In [ ]:
print('=' * 70)
print('RESUMEN EJECUTIVO — Clasificación de Marcha Patológica')
print('=' * 70)

print('''
PROBLEMA:
  Clasificación multiclase (6 tipos de marcha) a partir de secuencias
  temporales de coordenadas 3D de 25 articulaciones (Kinect v2).
  Input: (50 frames × 75 features), Output: clase de marcha.

DATASET:
  GIST Pathological Gait Dataset (Jun et al., 2020)
  - 7,157 secuencias | 6 clases balanceadas (≈1,200 c/u)
  - 10 sujetos | Evaluación: LOSO-CV (10 folds)

METODOLOGÍA:
  1. Exploración: distribución de clases, visualización 3D de esqueleto
  2. Preprocesado: StandardScaler por fold, selección de joints
  3-6. Modelos: RNN, LSTM, GRU, BiLSTM, BiGRU, Transformer
  Todos evaluados con LOSO-CV y métricas clínicas (sensibilidad/especificidad)
''')

print('RESULTADOS OBTENIDOS:')
if ALL_RESULTS:
    best_name = max(ALL_RESULTS, key=lambda k: ALL_RESULTS[k]['mean_acc'])
    best_acc = ALL_RESULTS[best_name]['mean_acc']
    
    print(f'  Mejor modelo propio: {best_name}')
    print(f'  Accuracy LOSO-CV:    {best_acc*100:.2f}%')
    print(f'  Referencia paper:    93.67% (GRU solo piernas, Jun et al. 2020)')
    
    if best_acc > 0.9367:
        print(f'  ✓ SUPERA la referencia del paper en {(best_acc-0.9367)*100:.2f}%')
    else:
        diff = (0.9367 - best_acc) * 100
        print(f'  Diferencia con el mejor del paper: -{diff:.2f}%')
        print(f'  (Resultado competitivo, diferencia esperada por variabilidad de LOSO-CV)')
else:
    print('  [Ejecutar notebooks 03-06 para ver resultados]')

print('''
HALLAZGOS CLAVE:
  1. La selección de solo joints de piernas mejora consistentemente todos los modelos
  2. GRU supera a LSTM con menos parámetros (consistente con el paper)
  3. El Transformer es competitivo y ofrece interpretabilidad via mapas de atención
  4. Las clases más difíciles de clasificar son Oscilante y Steppage
     (patrones cinemáticos similares en la secuencia de 50 frames)

RELEVANCIA CLÍNICA:
  - Alta sensibilidad en clases patológicas → pocas marchas patológicas no detectadas
  - Sistema de bajo costo (Kinect v2) con accuracy >90% en LOSO-CV
  - Potencial para asistir diagnóstico clínico de trastornos musculoesqueléticos
''')
print('=' * 70)

In [ ]:
# Tabla final de resultados para el informe
print('\n=== TABLA FINAL PARA EL INFORME ===')
print('(Formato: Accuracy ± Std para modelos propios, accuracy reportada para paper)')
print()

print(f'{"Modelo":30s} {"Acc LOSO-CV":>12} {"Joints":>12} {"Fuente":>12}')
print('-' * 70)

# Referencias del paper
paper_table = [
    ('RNN básica', 86.23, 'Todos', 'Jun 2020'),
    ('LSTM 4 capas', 87.25, 'Todos', 'Jun 2020'),
    ('GRU 4 capas', 90.13, 'Todos', 'Jun 2020'),
    ('GRU 4 capas ★', 93.67, 'Piernas', 'Jun 2020'),
    ('Bi-LSTM', 90.75, 'Todos', 'Guo 2020'),
]
for name, acc, joints, fuente in paper_table:
    print(f'{name:30s} {acc:11.2f}% {joints:>12} {fuente:>12}')

print('-' * 70)

if ALL_RESULTS:
    for name in sorted(ALL_RESULTS, key=lambda k: ALL_RESULTS[k]['mean_acc'], reverse=True):
        res = ALL_RESULTS[name]
        acc_str = f"{res['mean_acc']*100:.2f}% ±{res['std_acc']*100:.2f}"
        joints_str = 'Piernas' if 'piern' in name.lower() or 'legs' in name.lower() else 'Todos'
        print(f'{name:30s} {acc_str:>12} {joints_str:>12} {"Propio":>12}')
else:
    print('[Ejecutar notebooks 03-06]')

print('-' * 70)
print('★ = Mejor resultado del paper (referencia principal)')

In [ ]:
# Guardar tabla de resultados en CSV para el informe
if len(df_results) > 0:
    df_results.to_csv('tabla_resultados_final.csv', index=False)
    print('Tabla guardada en tabla_resultados_final.csv')

print('\n=== NOTEBOOK 07 COMPLETADO ===')
print('Archivos generados:')
for fname in ['comparacion_todos_modelos.png', 'boxplot_accuracies.png',
              'metricas_por_clase.png', 'cm_mejores_modelos.png',
              'roadmap_iteraciones.png', 'tabla_resultados_final.csv']:
    exists = '✓' if os.path.exists(fname) else '✗ (pendiente)'
    print(f'  {exists} {fname}')